# Tree growth: Most simple regression model

In this notebook, we will make the simplest possible regression model for the project, modeling growth based only on previous tree height and moisture level. 

<img src="http://mlsm.man.dtu.dk/mbml/wall_street.png">

The relevant imports.

In [159]:
import numpy as np
import pandas as pd   # We import Pandas!
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn import linear_model
import torch
import itertools

import libpysal
from esda.moran import Moran
from sklearn.metrics import r2_score

import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from pyro.infer import MCMC, NUTS, HMC, SVI, Trace_ELBO
from pyro.optim import Adam, ClippedAdam

# fix random generator seed (for reproducibility of results)
np.random.seed(42)

# matplotlib options
palette = itertools.cycle(sns.color_palette())
plt.style.use('ggplot')
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)

Lets start by loading the dataset. In order to let you focus on the probabilistic modelling aspects, we already prepared the raw GPS taxi data for you and extended it with additional information about the weather conditions.

In [160]:
# load csv (original dataset is by 30min intervals, we want 1h intervals) into a Pandas Dataframe
# df = pd.read_csv("https://mlsm.man.dtu.dk/mbml/pickups+weather_wallstreet.csv")

# look at the first few lines of the loaded dataset
# df.head()
df = pd.read_csv(
    "C:\\Users\\rasmu\\OneDrive - Danmarks Tekniske Universitet\\Dokumenter\\MBML project\\data\\out_10km_idx_preprocessed.csv",
    nrows=5000,          # start with a subset
    low_memory=False
)

df.head()


,x,y,biomassa_omdrev1,biomassa_omdrev2,flodesackumulering,grundyta_omdrev1,grundyta_omdrev2,markfuktighet,markfuktighet_klassad,medeldiameter_omdrev1,...,CenterLanNamn,CenterKommunNamn,is_no_forest,is_lake,delta_neg_medelhojd,delta_neg_p95,delta_neg_medeldiameter,delta_neg_biomassa,delta_neg_volym,is_stable_forest
0,445881.25,6260468.75,79.0,113.0,0.018730,22.0,23.0,97.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
1,445893.75,6260468.75,65.0,96.0,0.121874,17.0,20.0,99.0,2.0,18.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
2,445906.25,6260468.75,76.0,117.0,0.014839,21.0,24.0,99.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
3,445918.75,6260468.75,68.0,113.0,0.004559,19.0,23.0,85.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
4,445931.25,6260468.75,76.0,136.0,0.001676,21.0,27.0,82.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True


Filtering on "Is_stable_forest" 

In [161]:
# Keep only stable-forest rows
col_lookup = {c.lower(): c for c in df.columns}
stable_col = col_lookup.get("is_stable_forest")

# Fallback: find a likely stable-forest column if naming differs.
if stable_col is None:
    candidates = [
        c for c in df.columns
        if ("stable" in c.lower()) and ("forest" in c.lower())
    ]
    stable_col = candidates[0] if candidates else None

if stable_col is None:
    raise KeyError("No stable-forest column found (expected something like 'Is_stable_forest').")

stable_raw = df[stable_col]
if pd.api.types.is_bool_dtype(stable_raw):
    stable_mask = stable_raw
else:
    stable_mask = stable_raw.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

df = df.loc[stable_mask].copy()
print(f"Using stable-forest column: {stable_col}")
print(f"Rows after stable-forest filter: {len(df)}")
df.head()

Using stable-forest column: is_stable_forest
Rows after stable-forest filter: 2003


,x,y,biomassa_omdrev1,biomassa_omdrev2,flodesackumulering,grundyta_omdrev1,grundyta_omdrev2,markfuktighet,markfuktighet_klassad,medeldiameter_omdrev1,...,CenterLanNamn,CenterKommunNamn,is_no_forest,is_lake,delta_neg_medelhojd,delta_neg_p95,delta_neg_medeldiameter,delta_neg_biomassa,delta_neg_volym,is_stable_forest
0,445881.25,6260468.75,79.0,113.0,0.018730,22.0,23.0,97.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
1,445893.75,6260468.75,65.0,96.0,0.121874,17.0,20.0,99.0,2.0,18.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
2,445906.25,6260468.75,76.0,117.0,0.014839,21.0,24.0,99.0,2.0,15.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
3,445918.75,6260468.75,68.0,113.0,0.004559,19.0,23.0,85.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True
4,445931.25,6260468.75,76.0,136.0,0.001676,21.0,27.0,82.0,2.0,14.0,...,SKÅNE LÄN,OSBY,False,False,False,False,False,False,False,True


The model considers growth as a function of various forest characteristics including tree dimensions, volume, biomass, and soil wetness.



In [162]:
# prepare matrix with selected domain features (explicit list)
requested_cols = [
    "biomassa_omdrev1",
    "grundyta_omdrev1",
    "medeldiameter_omdrev1",
    "medelhojd_omdrev1",
    "p95_omdrev1",
    "vegetationskvot_omdrev1",
    "volym_omdrev1",
    "flodesackumulering",
    "markfuktighet",
]

available_cols = [c for c in requested_cols if c in df.columns]
missing_cols = [c for c in requested_cols if c not in df.columns]

if not available_cols:
    X_selected = np.empty((len(df), 0))
    X_weather = X_selected
    print("None of the requested columns were found in df.")
else:
    # Convert selected columns to numeric and handle missing values.
    X_selected = (
        df[available_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .values
    )

    # Keep existing downstream code compatible.
    X_weather = X_selected

    print(f"Using {len(available_cols)} requested columns")
    print(available_cols)
    if missing_cols:
        print(f"Missing {len(missing_cols)} requested columns")
        print(missing_cols)
    print(X_weather.shape)

Using 9 requested columns
['biomassa_omdrev1', 'grundyta_omdrev1', 'medeldiameter_omdrev1', 'medelhojd_omdrev1', 'p95_omdrev1', 'vegetationskvot_omdrev1', 'volym_omdrev1', 'flodesackumulering', 'markfuktighet']
(2003, 9)


In [163]:
def preprocess_species_clr(df, species_cols=None, eps=1e-9):
    if species_cols is None:
        species_cols = [
            "slu_skogskarta_biomassa",
            "slu_skogskarta_bjork_volym",
            "slu_skogskarta_bok_volym",
            "slu_skogskarta_contorta_volym",
            "slu_skogskarta_ek_volym",
            "slu_skogskarta_gran_volym",
            "slu_skogskarta_ovrigt_volym",
            "slu_skogskarta_tall_volym",
        ]

    present = [c for c in species_cols if c in df.columns]
    n_rows = len(df)
    n_sp = len(species_cols)

    if not present:
        perc = np.zeros((n_rows, n_sp), dtype=float)
        clr = np.zeros_like(perc)
        names = [c + "_clr" for c in species_cols]
        return perc, clr, names

    vals = df[present].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    row_sums = vals.sum(axis=1)

    # Percentages (0-100)
    perc = np.zeros_like(vals)
    nz = row_sums > 0
    if nz.any():
        perc[nz] = (vals[nz] / row_sums[nz][:, None]) * 100.0

    # CLR on proportions (use eps for zeros)
    clr = np.zeros_like(vals)
    for i in range(n_rows):
        if row_sums[i] <= 0:
            continue
        props = vals[i] / row_sums[i]
        props_safe = np.where(props <= 0, eps, props)
        gmean = np.exp(np.log(props_safe).mean())
        clr[i] = np.log(props_safe / gmean)

    # Reorder to full species_cols order and pad missing columns with zeros
    if len(present) != n_sp:
        perc_full = np.zeros((n_rows, n_sp), dtype=float)
        clr_full = np.zeros((n_rows, n_sp), dtype=float)
        for j, c in enumerate(species_cols):
            if c in present:
                idx = present.index(c)
                perc_full[:, j] = perc[:, idx]
                clr_full[:, j] = clr[:, idx]
        perc = perc_full
        clr = clr_full
        names = [c + "_clr" for c in species_cols]
    else:
        names = [c + "_clr" for c in present]

    return perc, clr, names

# Example usage (uncomment to run):
# perc, clr, clr_names = preprocess_species_clr(df)
# print('CLR shape:', clr.shape, 'CLR names:', clr_names)


In [164]:
# Integrate species CLR features into the feature matrix
perc, clr, clr_names = preprocess_species_clr(df)
print(f"Computed CLR matrix with shape: {clr.shape}")

# Append CLR columns to X_weather
if clr.size > 0:
    # Ensure X_weather is 2D numpy array
    if X_weather is None:
        X_weather = clr
    else:
        X_weather = np.hstack([X_weather, clr])
    print(f"Appended {clr.shape[1]} CLR features: {clr_names}")
else:
    print("No CLR features found or all species columns missing; nothing appended.")

# Keep a list of feature names (existing requested ones + CLR names) for reference
try:
    feature_names = available_cols + clr_names
except NameError:
    feature_names = clr_names

print(f"New X_weather shape: {X_weather.shape}")


Computed CLR matrix with shape: (2003, 8)
Appended 8 CLR features: ['slu_skogskarta_biomassa_clr', 'slu_skogskarta_bjork_volym_clr', 'slu_skogskarta_bok_volym_clr', 'slu_skogskarta_contorta_volym_clr', 'slu_skogskarta_ek_volym_clr', 'slu_skogskarta_gran_volym_clr', 'slu_skogskarta_ovrigt_volym_clr', 'slu_skogskarta_tall_volym_clr']
New X_weather shape: (2003, 17)


In [165]:
# Register the CLR columns in df so later spatial lag code can reference them directly.
# The CLR values are part of the model features even though they are derived columns.
if 'clr_names' not in globals():
    raise ValueError('clr_names is not available yet; run the CLR preprocessing cell first.')

clr_df = pd.DataFrame(clr, columns=clr_names, index=df.index)
df = pd.concat([df, clr_df], axis=1)

# Use one explicit feature-name list for all downstream model variants.
model_feature_cols = [c for c in available_cols + clr_names if c in df.columns]
print(f"Registered CLR columns in df: {clr_names}")
print(f"Model feature columns ({len(model_feature_cols)}): {model_feature_cols}")


Registered CLR columns in df: ['slu_skogskarta_biomassa_clr', 'slu_skogskarta_bjork_volym_clr', 'slu_skogskarta_bok_volym_clr', 'slu_skogskarta_contorta_volym_clr', 'slu_skogskarta_ek_volym_clr', 'slu_skogskarta_gran_volym_clr', 'slu_skogskarta_ovrigt_volym_clr', 'slu_skogskarta_tall_volym_clr']
Model feature columns (17): ['biomassa_omdrev1', 'grundyta_omdrev1', 'medeldiameter_omdrev1', 'medelhojd_omdrev1', 'p95_omdrev1', 'vegetationskvot_omdrev1', 'volym_omdrev1', 'flodesackumulering', 'markfuktighet', 'slu_skogskarta_biomassa_clr', 'slu_skogskarta_bjork_volym_clr', 'slu_skogskarta_bok_volym_clr', 'slu_skogskarta_contorta_volym_clr', 'slu_skogskarta_ek_volym_clr', 'slu_skogskarta_gran_volym_clr', 'slu_skogskarta_ovrigt_volym_clr', 'slu_skogskarta_tall_volym_clr']


In [166]:
# prepare target as growth in mean height between omdrev2 and omdrev1
y_raw = (
    pd.to_numeric(df["medelhojd_omdrev2"], errors="coerce")
    - pd.to_numeric(df["medelhojd_omdrev1"], errors="coerce")
)

# Fill any missing values before standardization.
y = y_raw.fillna(0.0).values

# standardize target
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / y_std
print(y.shape)
print(f"Target mean (before standardization): {y_mean:.4f}")
print(f"Target std (before standardization): {y_std:.4f}")

(2003,)
Target mean (before standardization): 35.1907
Target std (before standardization): 27.1310


The X matrix now contain all the input data for the model, and the y vector contains all the corresponding targets (growth).

The next step is to split our data into a train and test set. Alternatively, we could have used something like cross-validation, but for the sake of simplicity, a train/test split will do just fine for this example.

In [167]:
train_perc = 0.66  # percentage of training data
split_point = int(train_perc * len(y))
perm = np.random.permutation(len(y))
ix_train = perm[:split_point]
ix_test = perm[split_point:]

# Use the selected feature matrix as model input.
X = X_weather
X_train = X[ix_train, :]
X_test = X[ix_test, :]
y_train = y[ix_train]
y_test = y[ix_test]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("num train: %d" % len(y_train))
print("num test: %d" % len(y_test))

# ADD THIS:
# Standardize input features
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0) + 1e-8  # avoid division by zero
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std
print("\nFeatures standardized:")
print(f"X_train mean: {X_train.mean(axis=0)}")
print(f"X_train std: {X_train.std(axis=0)}")

print(f"y_train mean: {y_train.mean():.4f}")
print(f"y_train std: {y_train.std():.4f}")

X_train shape: (1321, 17)
X_test shape: (682, 17)
num train: 1321
num test: 682

Features standardized:
X_train mean: [-1.504e-17 -1.462e-17 -8.648e-17 -6.387e-18  1.415e-16  2.093e-17
  3.669e-17 -9.423e-18  4.538e-17  1.378e-17  1.354e-15 -2.752e-17
  1.316e-15  9.589e-16  2.808e-15  6.269e-16 -1.058e-15]
X_train std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
y_train mean: -0.0250
y_train std: 0.9297


A crucial step in developing a experimental setup in machine learning is establishing how to access the quality of the models that we learn. For this purpose, we developed a funciton which already constains a series of popular metrics for evaluating the quality of the predictions of a regression model (continuous output variables!). Here we just reuse the assesment from class, but it might be relevant to consider changing. 

In [168]:
def compute_error(trues, predicted):
    corr = np.corrcoef(predicted, trues)[0,1]
    mae = np.mean(np.abs(predicted - trues))
    rae = np.sum(np.abs(predicted - trues)) / np.sum(np.abs(trues - np.mean(trues)))
    rmse = np.sqrt(np.mean((predicted - trues)**2))
    r2 = max(0, 1 - np.sum((trues-predicted)**2) / np.sum((trues - np.mean(trues))**2))
    return corr, mae, rae, rmse, r2

We add some further metrics. 

In [ ]:
def evaluate_model(
    y_true,
    y_pred,
    coords=None,
    eps=1e-8
):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    residuals = y_pred - y_true

    metrics = {
        "MAE": np.mean(np.abs(residuals)),
        "RMSE": np.sqrt(np.mean(residuals**2)),
        "Bias": np.mean(residuals),
        "sMAPE": np.mean(2 * np.abs(residuals) / (np.abs(y_true) + np.abs(y_pred) + eps)),
        "R2": r2_score(y_true, y_pred),
        "Correlation": np.corrcoef(y_true, y_pred)[0, 1]
    }

    if coords is not None:
        coords = np.asarray(coords)
        if coords.ndim != 2 or coords.shape[1] != 2:
            raise ValueError(f"coords must be a 2D array with shape (n_samples, 2); got {coords.shape}")
        if coords.shape[0] != len(y_true):
            raise ValueError(f"coords must have the same number of rows as y_true/y_pred (got {coords.shape[0]} vs {len(y_true)})")

        from libpysal.weights import KNN
        from esda.moran import Moran

        w = KNN.from_array(coords, k=8)
        w.transform = "r"
        mi = Moran(residuals, w)

        metrics["Moran_I"] = mi.I
        metrics["Moran_p"] = mi.p_sim

    return metrics

I just added some data inspection for debugging and to see what's going on. The data has been standardized. 

In [170]:
# Inspect data quality
print("="*50)
print("DATA QUALITY CHECK")
print("="*50)
print("\nX_train stats:")
print(f"  Mean: {X_train.mean(axis=0)}")
print(f"  Std: {X_train.std(axis=0)}")
print(f"  Min: {X_train.min(axis=0)}")
print(f"  Max: {X_train.max(axis=0)}")

print("\ny_train stats:")
print(f"  Mean: {y_train.mean():.4f}, Std: {y_train.std():.4f}")

print("\nCorrelation with target:")
for i, col in enumerate(available_cols):
    corr = np.corrcoef(X_train[:, i], y_train)[0, 1]
    print(f"  {col}: {corr:.4f}")
print("="*50 + "\n")

DATA QUALITY CHECK

X_train stats:
  Mean: [-1.504e-17 -1.462e-17 -8.648e-17 -6.387e-18  1.415e-16  2.093e-17
  3.669e-17 -9.423e-18  4.538e-17  1.378e-17  1.354e-15 -2.752e-17
  1.316e-15  9.589e-16  2.808e-15  6.269e-16 -1.058e-15]
  Std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
  Min: [-1.204 -1.507 -1.532 -1.575 -1.503 -1.522 -1.101 -0.119 -1.463 -2.528
 -3.334 -1.344 -2.079 -1.396 -5.427 -1.4   -2.919]
  Max: [ 4.321  2.882  2.889  2.36   2.479  1.492  4.538 19.405  1.288  1.725
  1.456  4.515  2.537  2.696  1.744  2.624  1.576]

y_train stats:
  Mean: -0.0250, Std: 0.9297

Correlation with target:
  biomassa_omdrev1: -0.5011
  grundyta_omdrev1: -0.5338
  medeldiameter_omdrev1: -0.5831
  medelhojd_omdrev1: -0.5945
  p95_omdrev1: -0.5861
  vegetationskvot_omdrev1: -0.4880
  volym_omdrev1: -0.4767
  flodesackumulering: -0.0209
  markfuktighet: -0.1340



For the sake of comparision, we compare to linear regression from the sklearn package.

In [189]:
#regr = linear_model.LinearRegression()
regr = linear_model.Ridge()
regr.fit(X_train, y_train)
y_hat = regr.predict(X_test)

# Convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds)
print("\nNew Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


CorrCoef: 0.548
MAE: 13.550
RMSE: 25.558
R2: 0.295

New Evaluation Metrics:
MAE: 13.5498
RMSE: 25.5579
Bias: -1.9453
sMAPE: 0.3861
R2: 0.2953
Correlation: 0.5477



Better, right? But this took quite longer to run, and in this case didn't necessarily beat "sklearn"...

MCMC methods have great properties, namely the fact that in the limit of infinite computation time they will converge to the true posterior distribution. However, they often have difficulty scaling to larger datasets. 

### Pyro: Train on full dataset using Stochastic Variational Inference (SVI)

SVI on the other hand is much more scalable. Let us now try to use SVI to perform inference in our model. We start by preparing the data by converting it to torch tensors:

In [190]:
# Prepare data for Pyro model
X_train_torch = torch.tensor(X_train).float()
y_train_torch = torch.tensor(y_train).float()

In [191]:
def model(X, obs=None):
    alpha = pyro.sample("alpha", dist.Normal(0., 1.))
    
    beta  = pyro.sample("beta", dist.Normal(torch.zeros(X.shape[1]), 
                                            torch.ones(X.shape[1]) * 10.0).to_event())  # Weaker prior
    sigma = pyro.sample("sigma", dist.HalfCauchy(5.))
    
    with pyro.plate("data"):
        y = pyro.sample("y", dist.Normal(alpha + X.matmul(beta), sigma), obs=obs)
        
    return y

As we briefly touched upon during the notebook from last lecture (and also in the slides), in VI we specify a parametric distribution $q(\textbf{z}|\boldsymbol\phi)$ (called the *variational distribution*) with parameters $\boldsymbol\phi$. The goal is the to find the values of the parameters $\boldsymbol\phi$ that make $q(\textbf{z}|\boldsymbol\phi)$ and close as possible to the true posterior distribution $p(\textbf{z}|\textbf{x})$, thereby turning the problem of Bayesian inference into an optimization problem. We will get back to explain VI in detail later on in the course, but for now this is the essential that you need to know to use it in Pyro. 

In Pyro, the variational distribution is specified in a ```guide```, just like the model function. In fact, there are classes (e.g. ```AutoDiagonalNormal``` and ```AutoMultivariateNormal```) that generate the guide function automatically: 

In [192]:
# Define guide function
guide = AutoMultivariateNormal(model)

# Reset parameter values
pyro.clear_param_store()

Notice that we also reset the storage of Pyro parameters using ```pyro.clear_param_store()```. This is particularly important when running inference on the same model multiple times, because otherwise, Pyro will remember and re-use the parameter values from the last execution.

As mentioned above, in VI, the problem of (approximate) Bayesian inference is turned into an optimization problem. Like any other numerical optimization problem, we need to specify the optimizer (in this case ```ClippedAdam``` - a gradient descent algorithm that cleverly adapts the step size), an objective or loss function (```Trace_ELBO``` - this is the default that you will almost always use), and other details like the learning rate of the optimizer and the number of optimization steps:

In [193]:
# Define the number of optimization steps
n_steps = 4000

# Setup the optimizer
adam_params = {"lr": 0.001} # learning rate (lr) of optimizer
optimizer = ClippedAdam(adam_params)

# Setup the inference algorithm
elbo = Trace_ELBO(num_particles=1)
svi = SVI(model, guide, optimizer, loss=elbo)

The last step above was to instantiate ```SVI``` with the ```model```, ```guide```, ```optimizer``` and loss function (```elbo```) that we have just defined. Once this is done, we can solve this optimization problem by taking steps in the direction of the gradient using the function ```svi.step(...)```:

In [199]:
# Do gradient steps
for step in range(n_steps):
    elbo = svi.step(X_train_torch, y_train_torch)
    if step % 100 == 0:
        print("[%d] ELBO: %.1f" % (step, elbo))

[0] ELBO: 3642.1
[100] ELBO: 3606.7
[200] ELBO: 3622.9
[300] ELBO: 3451.2
[400] ELBO: 3437.4
[500] ELBO: 3476.2
[600] ELBO: 3375.3
[700] ELBO: 3334.9
[800] ELBO: 3325.9
[900] ELBO: 3215.9
[1000] ELBO: 3243.1
[1100] ELBO: 3036.4
[1200] ELBO: 3211.9
[1300] ELBO: 2970.9
[1400] ELBO: 3184.0
[1500] ELBO: 2918.5
[1600] ELBO: 2884.5
[1700] ELBO: 2884.2
[1800] ELBO: 2856.9
[1900] ELBO: 2792.3
[2000] ELBO: 2642.5
[2100] ELBO: 2704.9
[2200] ELBO: 2367.9
[2300] ELBO: 2566.1
[2400] ELBO: 2387.5
[2500] ELBO: 2484.6
[2600] ELBO: 2261.8
[2700] ELBO: 2179.5
[2800] ELBO: 2017.5
[2900] ELBO: 2049.9
[3000] ELBO: 1944.1
[3100] ELBO: 1849.5
[3200] ELBO: 1951.0
[3300] ELBO: 1820.1
[3400] ELBO: 1793.0
[3500] ELBO: 1730.6
[3600] ELBO: 1755.7
[3700] ELBO: 1677.3
[3800] ELBO: 1683.7
[3900] ELBO: 1718.2


If all went well, the values of the loss function in the output above should be going down. They can ocasionally go up, because Pyro estimates stochastic gradients (i.e. a noisy approximation to the gradients that, although imperfect is very efficient and scalable), but overall trend should be for the values of the loss function to go down. 

Once the optimization has converged to a minimum, we can use the ```Predictive``` class to extract the results. The usage is similar to what we did with MCMC, although we now must pass the guide to the ```Predictive``` class as well:

In [200]:
from pyro.infer import Predictive

predictive = Predictive(model, guide=guide, num_samples=1000,
                        return_sites=("alpha", "beta", "sigma"))
samples = predictive(X_train_torch, y_train_torch)

We can now use the samples from the posterior distribution above to make predictions for the test set:

In [201]:
alpha_samples = samples["alpha"].detach().numpy().ravel()  # Force 1D
beta_samples = samples["beta"].detach().numpy()

# Determine number of samples and features
n_samples = len(alpha_samples)
n_features = X_test.shape[1]

# Reshape beta to (n_samples, n_features)
beta_samples = beta_samples.reshape(n_samples, n_features)

# Compute predictions for all samples at once
# X_test is (682, 9), beta_samples is (1000, 9)
# X_test @ beta_samples.T gives (682, 1000)
# Add alpha to each column (broadcasting)
preds_all = X_test @ beta_samples.T + alpha_samples[np.newaxis, :]
# Average across samples
y_hat = preds_all.mean(axis=1)

In [202]:
# Debug: check shapes before prediction
print("Raw alpha_samples shape:", samples["alpha"].detach().numpy().shape)
print("Raw beta_samples shape:", samples["beta"].detach().numpy().shape)
print("X_test shape:", X_test.shape)
print()

# Extract and flatten
alpha_samples = samples["alpha"].detach().numpy().flatten()
beta_samples = samples["beta"].detach().numpy()
print("After flatten - alpha shape:", alpha_samples.shape)
print("After flatten - beta shape:", beta_samples.shape)
print()

# Reshape beta
beta_samples = beta_samples.reshape(1000, -1)
print("After reshape - beta shape:", beta_samples.shape)
print("beta.T shape:", beta_samples.T.shape)

Raw alpha_samples shape: (1000, 1)
Raw beta_samples shape: (1000, 1, 17)
X_test shape: (682, 17)

After flatten - alpha shape: (1000,)
After flatten - beta shape: (1000, 1, 17)

After reshape - beta shape: (1000, 17)
beta.T shape: (17, 1000)


In [ ]:
alpha_samples = samples["alpha"].detach().numpy()
beta_samples = samples["beta"].detach().numpy()
y_hat = np.mean(alpha_samples.T + np.dot(X_test, beta_samples[:,0].T), axis=1)

# convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds)
print("\nSVI New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

CorrCoef: 0.464
MAE: 14.929
RMSE: 27.289
R2: 0.197

New Evaluation Metrics:
MAE: 14.9290
RMSE: 27.2888
Bias: -1.9810
sMAPE: 0.4345
R2: 0.1966
Correlation: 0.4640


In [210]:
# === Exact conjugate Bayesian linear regression (Normal-Inverse-Gamma) ===
# This cell computes the analytical posterior (Normal-InvGamma) and posterior predictive samples.
import numpy as np

def exact_bayesian_linear_regression(X_train, y_train, X_test=None, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2):
    """Posterior sampling for linear regression with conjugate Normal-Inverse-Gamma prior.

    Prior: theta | sigma2 ~ N(0, sigma2 * V0),  sigma2 ~ InvGamma(a0, b0)
    V0 = tau^2 * I provides a weak (std=tau) prior on coefficients.
    Returns posterior samples and posterior predictive mean (if X_test provided).
    """
    n, p = X_train.shape
    X_train_design = np.hstack([np.ones((n,1)), X_train])
    if X_test is not None:
        X_test_design = np.hstack([np.ones((X_test.shape[0],1)), X_test])
    else:
        X_test_design = None

    p_design = p + 1
    mu0 = np.zeros(p_design)
    V0 = (tau ** 2) * np.eye(p_design)
    V0_inv = np.linalg.inv(V0)

    XtX = X_train_design.T @ X_train_design
    Xty = X_train_design.T @ y_train

    Vn_inv = XtX + V0_inv
    Vn = np.linalg.inv(Vn_inv)
    mu_n = Vn @ Xty

    a_n = a0 + n / 2.0
    term = (y_train.T @ y_train) + (mu0.T @ V0_inv @ mu0) - (mu_n.T @ Vn_inv @ mu_n)
    b_n = b0 + 0.5 * term

    # Sample sigma2 via gamma relation and then theta conditional on sigma2
    gamma_draws = np.random.gamma(shape=a_n, scale=1.0 / b_n, size=n_samples)
    sigma2_samples = 1.0 / gamma_draws

    L = np.linalg.cholesky(Vn)
    theta_samples = np.empty((n_samples, p_design))
    for i, s2 in enumerate(sigma2_samples):
        z = np.random.normal(size=p_design)
        theta_samples[i] = mu_n + np.sqrt(s2) * (L @ z)

    if X_test_design is not None:
        preds = X_test_design @ theta_samples.T  # (n_test, n_samples)
        y_hat_mean = preds.mean(axis=1)
        return theta_samples, sigma2_samples, y_hat_mean
    return theta_samples, sigma2_samples


# Run exact inference and evaluate
theta_samps, sigma2_samps, y_hat_exact = exact_bayesian_linear_regression(X_train, y_train, X_test, n_samples=1000, tau=10.0, a0=1e-2, b0=1e-2)
print('theta_samples shape:', theta_samps.shape)
print('sigma2_samples shape:', sigma2_samps.shape)

preds_exact = y_hat_exact * y_std + y_mean
y_true = y_test * y_std + y_mean
corr, mae, rae, rmse, r2 = compute_error(y_true, preds_exact)
print('\nExact conjugate posterior predictive results:')
print('CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f' % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds_exact)
print("\n Exact Inference New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

theta_samples shape: (1000, 18)
sigma2_samples shape: (1000,)

Exact conjugate posterior predictive results:
CorrCoef: 0.548
MAE: 13.541
RMSE: 25.552
R2: 0.296

 Exact Inference New Evaluation Metrics:
MAE: 13.5408
RMSE: 25.5519
Bias: -1.9673
sMAPE: 0.3856
R2: 0.2957
Correlation: 0.5480


## Spatial lag covariates

As a first step toward modeling correlation between neighboring cells, we can add a spatial lag term by averaging each predictor over the immediate rook neighbors of each cell. This keeps the model linear and lets us reuse the same Bayesian machinery.


In [206]:
# Build spatial lag covariates from immediate rook neighbours on the regular grid.
# This keeps the model linear: x_i is augmented with the mean of the same predictors in N(i).
# We now lag the explicit model_feature_cols list, which includes the CLR columns we registered in df.

requested_spatial_cols = list(dict.fromkeys(model_feature_cols)) if "model_feature_cols" in globals() else []
if not requested_spatial_cols:
    raise ValueError("model_feature_cols is not available yet; run the feature-construction cells first.")

spatial_base_cols = [c for c in requested_spatial_cols if c in df.columns]
missing_spatial_cols = [c for c in requested_spatial_cols if c not in df.columns]
if missing_spatial_cols:
    print("Skipping columns that are not in df and cannot be lagged directly:")
    print(missing_spatial_cols)

if not spatial_base_cols:
    raise ValueError("No spatial base columns exist in df after filtering; cannot build spatial lag features.")

if "x" not in df.columns or "y" not in df.columns:
    raise KeyError("Spatial lag features require x and y coordinate columns in df.")

# Preserve coordinates and predictors; keep coordinates numeric and rounded to avoid join noise.
spatial_df = df[["x", "y"] + spatial_base_cols].copy()
spatial_df[["x", "y"]] = spatial_df[["x", "y"]].apply(pd.to_numeric, errors="coerce")
spatial_df[spatial_base_cols] = spatial_df[spatial_base_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
spatial_df[["x", "y"]] = spatial_df[["x", "y"]].round(6)

# Infer the grid spacing from the observed coordinates.
x_vals = np.sort(spatial_df["x"].unique())
y_vals = np.sort(spatial_df["y"].unique())
x_step = np.diff(x_vals)
y_step = np.diff(y_vals)
step_candidates = np.concatenate([x_step[x_step > 0], y_step[y_step > 0]]) if (len(x_step) or len(y_step)) else np.array([])
if step_candidates.size == 0:
    raise ValueError("Could not infer a positive grid spacing from x/y coordinates.")
cell_step = float(np.min(step_candidates))

coord_df = spatial_df[["x", "y"]].copy()
lag_sum = np.zeros((len(spatial_df), len(spatial_base_cols)), dtype=float)
lag_count = np.zeros(len(spatial_df), dtype=float)
offsets = [(-cell_step, 0.0), (cell_step, 0.0), (0.0, -cell_step), (0.0, cell_step)]

for dx, dy in offsets:
    neighbor = spatial_df[["x", "y"] + spatial_base_cols].copy()
    neighbor["x"] = (neighbor["x"] - dx).round(6)
    neighbor["y"] = (neighbor["y"] - dy).round(6)

    # Make the lagged columns explicit before the merge so we do not depend on pandas suffix rules.
    neighbor = neighbor.rename(columns={c: f"{c}_nbr" for c in spatial_base_cols})
    merged = coord_df.merge(neighbor, on=["x", "y"], how="left")

    neighbor_cols = [f"{c}_nbr" for c in spatial_base_cols]
    available = merged[neighbor_cols[0]].notna().to_numpy()
    lag_count += available.astype(float)
    lag_sum += merged[neighbor_cols].fillna(0.0).to_numpy()

lag_mean = np.divide(
    lag_sum,
    lag_count[:, None],
    out=np.zeros_like(lag_sum),
    where=lag_count[:, None] > 0,
)

spatial_lag_cols = [f"{c}_lag" for c in spatial_base_cols]
X_spatial = np.hstack([spatial_df[spatial_base_cols].to_numpy(), lag_mean])
X_spatial_names = spatial_base_cols + spatial_lag_cols

print(f"Inferred grid spacing: {cell_step:g}")
print(f"Spatial lag matrix shape: {X_spatial.shape}")
print(f"Base features: {len(spatial_base_cols)}")
print(f"Lag features: {len(spatial_lag_cols)}")
print(f"Example lag feature names: {spatial_lag_cols[:5]}")


Inferred grid spacing: 12.5
Spatial lag matrix shape: (2003, 34)
Base features: 17
Lag features: 17
Example lag feature names: ['biomassa_omdrev1_lag', 'grundyta_omdrev1_lag', 'medeldiameter_omdrev1_lag', 'medelhojd_omdrev1_lag', 'p95_omdrev1_lag']


In [209]:
# Spatial lag exact Bayesian linear regression
# We reuse the conjugate exact solver above on the expanded design matrix.

X_spatial_train = X_spatial[ix_train, :]
X_spatial_test = X_spatial[ix_test, :]

# Standardize the expanded feature set separately from the base model.
X_spatial_mean = X_spatial_train.mean(axis=0)
X_spatial_std = X_spatial_train.std(axis=0) + 1e-8
X_spatial_train_std = (X_spatial_train - X_spatial_mean) / X_spatial_std
X_spatial_test_std = (X_spatial_test - X_spatial_mean) / X_spatial_std

print(f"X_spatial_train shape: {X_spatial_train_std.shape}")
print(f"X_spatial_test shape: {X_spatial_test_std.shape}")

# Exact Bayesian inference for the expanded model.
theta_spatial_samps, sigma2_spatial_samps, y_hat_spatial = exact_bayesian_linear_regression(
    X_spatial_train_std,
    y_train,
    X_spatial_test_std,
    n_samples=1000,
    tau=10.0,
    a0=1e-2,
    b0=1e-2,
)

print("theta_spatial_samps shape:", theta_spatial_samps.shape)
print("sigma2_spatial_samps shape:", sigma2_spatial_samps.shape)

preds_spatial_exact = y_hat_spatial * y_std + y_mean
y_true_spatial = y_test * y_std + y_mean
corr, mae, rae, rmse, r2 = compute_error(y_true_spatial, preds_spatial_exact)
print("\nSpatial-lag exact Bayesian results:")
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true_spatial, preds_spatial_exact, coords=df.iloc[ix_test][["x", "y"]].to_numpy())
print("\nSpatial New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

X_spatial_train shape: (1321, 34)
X_spatial_test shape: (682, 34)
theta_spatial_samps shape: (1000, 35)
sigma2_spatial_samps shape: (1000,)

Spatial-lag exact Bayesian results:
CorrCoef: 0.531
MAE: 13.533
RMSE: 25.898
R2: 0.276

Spatial New Evaluation Metrics:
MAE: 13.5332
RMSE: 25.8983
Bias: -2.3453
sMAPE: 0.3917
R2: 0.2764
Correlation: 0.5314
Moran_I: 0.3889
Moran_p: 0.0010


c:\Users\rasmu\miniconda3\envs\mbml\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


## 2.3 Pyro: Heteroscedastic regression

Ok, let us now assume again that the Gaussian likelihood was indeed the most appropriate choice. In many problems of interest, it is often the case that constant observation variance ($\sigma^2$) is too limiting or inadequate. We can relax this assumption by considering heteroscedastic models, in which the observation variance is assumed to be non-constant and dependent on any other variables. In this particular case, we shall assume that the observation variance is also linearly dependent on the inputs $\textbf{x}$. 

Lets implement this model in Pyro (check the lecture slides, if necessary)!

In [62]:
def heteroscedastic_model(X, obs=None):
    alpha_mu = pyro.sample("alpha_mu", dist.Normal(0., 1.))                 # Prior for the bias/intercept of the mean
    beta_mu  = pyro.sample("beta_mu", dist.Normal(torch.zeros(X.shape[1]), 
                                               torch.ones(X.shape[1])).to_event())     # Priors for the regression coeffcients of the mean
    alpha_v = pyro.sample("alpha_v", dist.Normal(0., 1.))                   # Prior for the bias/intercept of the variance
    beta_v  = pyro.sample("beta_v", dist.Normal(torch.zeros(X.shape[1]), 
                                               torch.ones(X.shape[1])).to_event())     # Priors for the regression coeffcients of the variance
    
    with pyro.plate("data"):
        y = pyro.sample("y", dist.Normal(alpha_mu + X.matmul(beta_mu), torch.exp(alpha_v + X.matmul(beta_v))), obs=obs)
        
    return y

Once you finished coding the model, it is time to run inference on it. Feel free to play with the hyper-parameters of the optimization (i.e. ```n_steps```, learning rate ```lr```, etc.):

In [63]:
# Prepare data for Pyro model
X_train_torch = torch.tensor(X_train).float()
y_train_torch = torch.tensor(y_train).float()

In [64]:
# Define guide function
guide = AutoMultivariateNormal(heteroscedastic_model)

# Reset parameter values
pyro.clear_param_store()

# Define the number of optimization steps
n_steps = 8000

# Setup the optimizer
adam_params = {"lr": 0.001} # learning rate (lr) of optimizer
optimizer = ClippedAdam(adam_params)

# Setup the inference algorithm
elbo = Trace_ELBO(num_particles=1)
svi = SVI(heteroscedastic_model, guide, optimizer, loss=elbo)

# Do gradient steps
for step in range(n_steps):
    elbo = svi.step(X_train_torch, y_train_torch)
    if step % 200 == 0:
        print("[%d] ELBO: %.1f" % (step, elbo))

[0] ELBO: 6230.4
[200] ELBO: 2484.8
[400] ELBO: 2073.7
[600] ELBO: 1611.2
[800] ELBO: 1629.0
[1000] ELBO: 1453.1
[1200] ELBO: 1331.3
[1400] ELBO: 1480.0
[1600] ELBO: 1299.5
[1800] ELBO: 1317.5
[2000] ELBO: 1308.0
[2200] ELBO: 1324.8
[2400] ELBO: 1313.9
[2600] ELBO: 1289.5
[2800] ELBO: 1247.5
[3000] ELBO: 1257.0
[3200] ELBO: 1321.2
[3400] ELBO: 1265.8
[3600] ELBO: 1239.5
[3800] ELBO: 1260.2
[4000] ELBO: 1242.4
[4200] ELBO: 1223.0
[4400] ELBO: 1240.0
[4600] ELBO: 1236.1
[4800] ELBO: 1230.0
[5000] ELBO: 1227.2
[5200] ELBO: 1227.6
[5400] ELBO: 1224.4
[5600] ELBO: 1232.3
[5800] ELBO: 1225.7
[6000] ELBO: 1219.4
[6200] ELBO: 1218.8
[6400] ELBO: 1214.0
[6600] ELBO: 1220.2
[6800] ELBO: 1221.6
[7000] ELBO: 1210.5
[7200] ELBO: 1226.3
[7400] ELBO: 1224.1
[7600] ELBO: 1214.2
[7800] ELBO: 1212.4


We can now extract the samples from the posterior distribution and use them to make predictions for the test set:

In [65]:
from pyro.infer import Predictive

predictive = Predictive(heteroscedastic_model, guide=guide, num_samples=1000,
                        return_sites=("alpha_mu", "beta_mu", "alpha_v", "beta_v"))
samples = predictive(X_train_torch, y_train_torch)

In [ ]:
alpha_samples = samples["alpha_mu"].detach().numpy()
beta_samples = samples["beta_mu"].detach().numpy()
y_hat = np.mean(alpha_samples.T + np.dot(X_test, beta_samples[:,0].T), axis=1)

# convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

metrics = evaluate_model(y_true, preds)
print("\nHeteroscedastic New Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

CorrCoef: 0.556
MAE: 13.575
RMSE: 25.420
R2: 0.303


In [67]:
alpha_v_samples = samples["alpha_v"].detach().numpy()
beta_v_samples = samples["beta_v"].detach().numpy()
sigma_hat = np.mean(np.exp(alpha_v_samples.T + np.dot(X_test, beta_v_samples[:,0].T)), axis=1)

In [68]:
np.set_printoptions(precision=3)
print(sigma_hat[:10])

[0.398 0.477 0.326 0.337 0.387 0.4   1.341 0.368 0.354 0.555]


Notice how it changes over time. We can use these values to estimate prediction intervals (e.g. 95% prediction intervals) that are time-varying. Can you think of real-world problems where this information would be useful?